##importing functions

In [0]:
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.functions import when
from pyspark.sql.functions import length,trim
from pyspark.sql.functions import min  , max
from pyspark.sql.functions import to_date, to_timestamp
from pyspark.sql.functions import date_format
print(' functions imported succesfully')

#reading table

In [0]:
df=spark.read.table('workspace.bronze.orders')
print('read table succesfully!')

##Bronze Layer validation

In [0]:
print('checking schema ...')
df.printSchema()
print('checking data types ...')
print(df.dtypes)
print('checking as sample of data')
df.show(3)

##column operations

In [0]:
#naming convention is snake_case no need to change
#no bad data types but change as defensive approach
df = df.withColumn('order_id', col("order_id").cast('integer')).withColumn('date', col("date").cast('date')).withColumn('time', date_format(col("time"), "HH:mm:ss"))
print('changed data types successfully!')

#check for trailling spaces,trimming ##no need to trim no string column
#check max,min order_id 
df.select(max('order_id')).show()
df.select(min('order_id')).show()
df.select('order_id').count()



##row operations

In [0]:
#handle duplicates
df_unique=df.distinct()
if df_unique.count() != df.count():
    print('found duplicates')
    print('original row count',df.count())
    print('unique row count',df_unique.count())
    df=df.distinct()
    print('duplicates handled successfully')
else :
    print('no duplicates found')
    print('duplicates checked successfully!')

print('-'*40)
#handle nulls
df_anynulls=df.na.drop('any')
df_allnulls=df.na.drop('all')
if df_anynulls.count() != df.count() or df_allnulls.count() != df.count():
    print('nulls found')
    print('original row count',df.count())
    print('rows count after removing rows containing ANY nulls',df_anynulls.count())
    print('rows count after removing rows containing ALL nulls',df_allnulls.count())
    print('action needed')
else :
    print('no nulls found')
    print('nulls checked successfully!')

print('-'*40)


##validation

In [0]:
print('-'*40)
print('Data validation checks')
print('-'*40)
validation_results=[]
def validate(colname,rule,condition) :
    invalid= df.filter(condition).count()
    validation_results.append({
        'column' : colname,
        'rule' : rule,
        'invalid_count' : invalid,
        'invalid_pct': round(invalid / df.count() * 100, 2)
    })

validate('order_id','order_id > 0',col('order_id') <= 0)
validate('date','invalid date',to_date(col('date')).isNull() )
validate('time','invalid time',to_timestamp(col('time')).isNull())

spark.createDataFrame(validation_results).show(truncate=False)


##saving

In [0]:
#save as delta 
df.write.mode('overwrite').format('delta').option("overwriteSchema", "true").saveAsTable('workspace.silver.orders')
print('saved...')

In [0]:
%sql
--sanity check
SELECT * FROM workspace.silver.orders LIMIT 10